# 01 — DNABERT-2 Embedding Extraction & Caching

**Purpose**: Extract hidden states from all 13 layers of DNABERT-2, apply 3 pooling strategies, cache to disk.

**Reproducibility notes**:
- DNABERT-2 uses `BertUnpadAttention` which flattens non-padding tokens internally.
- We use `register_forward_hook` to capture each layer's output, then `repad()` to restore `[batch, seq_len, hidden_dim]` shape.
- `output_hidden_states=True` does NOT work with this custom model — hooks are required.
- `DataLoader(shuffle=False)` is critical to keep embedding–label alignment.

**Output**: `./embedding_cache/` containing 84 `.npy` files (42 train + 42 test).

| Layer key | Source |
|-----------|--------|
| 0 | `model.embeddings` output |
| 1–12 | `model.encoder.layer[0]`–`[11]` output |
| avg_last4 | mean of layers 9, 10, 11, 12 |

## 0. Install Dependencies

In [1]:
# Pin exact versions for reproducibility
!pip install einops transformers==4.40.0 peft omegaconf evaluate accelerate scikit-learn genomic-benchmarks

# After install, record all versions
import transformers, torch, numpy, sklearn
print(f'torch:        {torch.__version__}')
print(f'transformers: {transformers.__version__}')
print(f'numpy:        {numpy.__version__}')
print(f'sklearn:      {sklearn.__version__}')

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.


  Using cached transformers-4.40.0-py3-none-any.whl.metadata (137 kB)
  Using cached tokenizers-0.19.1-cp310-none-win_amd64.whl.metadata (6.9 kB)
Using cached transformers-4.40.0-py3-none-any.whl (9.0 MB)
Using cached tokenizers-0.19.1-cp310-none-win_amd64.whl (2.2 MB)

  Attempting uninstall: tokenizers

    Found existing installation: tokenizers 0.22.2

    Uninstalling tokenizers-0.22.2:

      Successfully uninstalled tokenizers-0.22.2

   ---------------------------------------- 0/2 [tokenizers]
   ---------------------------------------- 0/2 [tokenizers]
   ---------------------------------------- 0/2 [tokenizers]
   ---------------------------------------- 0/2 [tokenizers]
   ---------------------------------------- 0/2 [tokenizers]
   ---------------------------------------- 0/2 [tokenizers]
   ---------------------------------------- 0/2 [tokenizers]
   ---------------------------------------- 0/2 [tokenizers]
   ---------------------------------------- 0/2 [tokenizers]
   --

## 1. Config

In [2]:
import torch
import torch.nn as nn
import numpy as np
import os
import hashlib
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from genomic_benchmarks.dataset_getters.pytorch_datasets import get_dataset
from tqdm import tqdm

# ============ CONFIG ============
MODEL_DIR   = './dnabert2'         # local path after snapshot_download
MODEL_REPO  = 'zhihan1996/DNABERT-2-117M'  # HuggingFace repo for download
CACHE_DIR   = './embedding_cache'
BATCH_SIZE  = 32
MAX_LENGTH  = 512
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# ================================

print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

Device: cuda
GPU: NVIDIA GeForce RTX 5090
Memory: 31.8 GB


## 2. Download & Load Model

In [3]:
# Download model if not already present
if not os.path.exists(MODEL_DIR):
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id=MODEL_REPO,
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False
    )
    print(f'Downloaded model to {MODEL_DIR}')
else:
    print(f'Model already exists at {MODEL_DIR}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
model = AutoModel.from_pretrained(MODEL_DIR, trust_remote_code=True)
model = model.to(DEVICE)
model.eval()

num_layers = model.config.num_hidden_layers   # 12
hidden_dim = model.config.hidden_size          # 768
print(f'Transformer layers: {num_layers}')
print(f'Hidden dim: {hidden_dim}')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

# Record model config for reproducibility
print(f'\n--- Model Config ---')
print(f'Model repo:     {MODEL_REPO}')
print(f'Vocab size:     {model.config.vocab_size}')
print(f'Max position:   {model.config.max_position_embeddings}')
print(f'Num attention heads: {model.config.num_attention_heads}')

E:\anaconda\envs\nlp_dl\lib\site-packages\huggingface_hub\file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/468M [00:00<?, ?B/s]

Downloaded model to ./dnabert2


C:\Users\JxxC\.cache\huggingface\modules\transformers_modules\dnabert2\bert_layers.py:126: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at ./dnabert2 and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Transformer layers: 12
Hidden dim: 768
Total parameters: 117,068,544

--- Model Config ---
Model repo:     zhihan1996/DNABERT-2-117M
Vocab size:     4096
Max position:   512
Num attention heads: 12


## 3. Load Dataset

In [4]:
DATASET_NAME = 'human_nontata_promoters'

train_gb = get_dataset(DATASET_NAME, split='train')
test_gb  = get_dataset(DATASET_NAME, split='test')

print(f'Dataset: {DATASET_NAME}')
print(f'Train samples: {len(train_gb)}')
print(f'Test samples:  {len(test_gb)}')

seq, label = train_gb[0]
print(f'\nExample:')
print(f'  Sequence length: {len(seq)}')
print(f'  Label: {label}')
print(f'  Preview: {seq[:60]}...')

# Check label distribution
train_labels_check = [l for _, l in train_gb]
test_labels_check  = [l for _, l in test_gb]
print(f'\nTrain label distribution: {np.bincount(train_labels_check)}')
print(f'Test  label distribution: {np.bincount(test_labels_check)}')

Dataset: human_nontata_promoters
Train samples: 27097
Test samples:  9034

Example:
  Sequence length: 251
  Label: 0
  Preview: TCGGCCTGCGGACTCTTCCGGCCCGAACTTCTCTTACCTACCCCGCGCTCCCCGGTGCAG...

Train label distribution: [12355 14742]
Test  label distribution: [4119 4915]


In [5]:
class DnaDataset(Dataset):
    def __init__(self, base_dataset, tokenizer, max_length=MAX_LENGTH):
        self.base_dataset = base_dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        sequence, label = self.base_dataset[idx]
        encoding = self.tokenizer(
            sequence,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(label, dtype=torch.long)
        return item

train_dataset = DnaDataset(train_gb, tokenizer)
test_dataset  = DnaDataset(test_gb, tokenizer)

# ★ shuffle=False: CRITICAL for embedding-label alignment
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Train batches: {len(train_loader)}')
print(f'Test batches:  {len(test_loader)}')

# Verify tokenization
sample = train_dataset[0]
non_pad = (sample['attention_mask'] == 1).sum().item()
print(f'\nSample: {non_pad} non-padding tokens out of {MAX_LENGTH}')

Train batches: 847
Test batches:  283

Sample: 59 non-padding tokens out of 512


## 4. Pooling Functions

| Method | Description | Literature |
|--------|-------------|------------|
| mean | Average all non-padding token embeddings | Best overall in Feng et al. (Nature Comm 2025) |
| max | Element-wise max over non-padding tokens | — |
| cls | First token embedding ("summary token") | Standard BERT approach |

In [6]:
def mean_pool(hidden_states, attention_mask):
    """
    Mean pooling with attention mask.
    hidden_states: [batch, seq_len, hidden_dim]
    attention_mask: [batch, seq_len]
    Returns: [batch, hidden_dim]
    """
    mask = attention_mask.unsqueeze(-1).float()            # [batch, seq_len, 1]
    masked = hidden_states * mask                           # zero out padding
    summed = masked.sum(dim=1)                              # [batch, hidden_dim]
    counts = mask.sum(dim=1).clamp(min=1e-9)                # avoid div by zero
    return summed / counts


def max_pool(hidden_states, attention_mask):
    """
    Max pooling — padding positions set to -inf before max.
    """
    mask = attention_mask.unsqueeze(-1).float()
    h = hidden_states.clone()
    h[mask.squeeze(-1) == 0] = float('-inf')
    return h.max(dim=1)[0]


def cls_pool(hidden_states, attention_mask):
    """
    CLS / summary token — first token embedding.
    DNABERT-2 uses BPE (no explicit [CLS]); first token serves as summary.
    """
    return hidden_states[:, 0, :]


POOLING_FNS = {
    'mean': mean_pool,
    'max':  max_pool,
    'cls':  cls_pool,
}

print('Pooling methods:', list(POOLING_FNS.keys()))

Pooling methods: ['mean', 'max', 'cls']


## 5. Re-padding Utility

**Why this is needed**: DNABERT-2 uses `BertUnpadAttention` which internally removes padding tokens and flattens all non-padding tokens into `[total_tokens, hidden_dim]`. The encoder layers' hook outputs have this flattened shape, not the expected `[batch, seq_len, hidden_dim]`. We must re-pad before pooling.

In [7]:
def repad(unpadded, attention_mask):
    """
    Re-pad unpadded encoder output back to [batch, seq_len, hidden_dim].
    
    DNABERT-2's BertUnpadAttention strips padding internally, so encoder
    layer outputs are [total_non_pad_tokens, hidden_dim] instead of
    [batch, seq_len, hidden_dim]. This function restores the standard shape.
    
    Args:
        unpadded:        [total_non_pad_tokens, hidden_dim]
        attention_mask:  [batch, seq_len]
    Returns:
        padded:          [batch, seq_len, hidden_dim] (padding positions = 0)
    """
    batch_size, seq_len = attention_mask.shape
    hidden_dim = unpadded.shape[-1]
    
    padded = torch.zeros(batch_size, seq_len, hidden_dim, device=unpadded.device)
    lengths = attention_mask.sum(dim=1).long()
    
    offset = 0
    for i in range(batch_size):
        l = lengths[i].item()
        padded[i, :l] = unpadded[offset:offset+l]
        offset += l
    
    return padded

print('repad() defined')

repad() defined


## 6. Extraction Function (Hook-based)

**Design**:
- Register `forward_hook` on `model.embeddings` (layer 0) and `model.encoder.layer[i]` (layers 1–12)
- Single forward pass captures all 13 layers
- Apply 3 poolings per layer → 39 embeddings + 3 for avg_last4 = 42 per sample
- Hooks are cleaned up after extraction

In [8]:
def extract_all_embeddings(data_loader, model, device, num_layers=12, desc='Extracting'):
    """
    Extract embeddings from all layers with all pooling methods using forward hooks.
    
    Handles DNABERT-2's unpadding: encoder layers output [total_tokens, 768]
    which we re-pad to [batch, seq_len, 768] before pooling.
    
    Returns:
        embeddings: dict[layer_key][pool_name] -> np.array [num_samples, hidden_dim]
        labels:     np.array [num_samples]
    
    Layer keys: 0 (embedding), 1-12 (transformer), 'avg_last4'
    """
    LAYER_INDICES = list(range(num_layers + 1))  # [0, 1, ..., 12]
    
    # --- Register hooks ---
    captured = {}  # {layer_idx: tensor} — refreshed each forward pass
    hooks = []
    
    def make_hook(layer_idx):
        def hook_fn(module, input, output):
            if isinstance(output, tuple):
                captured[layer_idx] = output[0]
            else:
                captured[layer_idx] = output
        return hook_fn
    
    # Hook embedding layer → layer 0
    hooks.append(model.embeddings.register_forward_hook(make_hook(0)))
    
    # Hook each transformer layer → layers 1–12
    for i in range(num_layers):
        hooks.append(model.encoder.layer[i].register_forward_hook(make_hook(i + 1)))
    
    # --- Storage ---
    all_emb = {}
    for li in LAYER_INDICES:
        all_emb[li] = {name: [] for name in POOLING_FNS}
    all_emb['avg_last4'] = {name: [] for name in POOLING_FNS}
    all_labels = []
    
    # --- Forward pass ---
    for batch in tqdm(data_loader, desc=desc):
        input_ids      = batch['input_ids'].to(device)
        attention_mask  = batch['attention_mask'].to(device)
        labels          = batch['labels']
        
        captured.clear()
        
        with torch.no_grad():
            _ = model(input_ids=input_ids, attention_mask=attention_mask)
        
        # Per-layer pooling
        for li in LAYER_INDICES:
            h = captured[li]
            # Re-pad if encoder returned unpadded [total_tokens, 768]
            if h.dim() == 2:
                h = repad(h, attention_mask)
            for pname, pfn in POOLING_FNS.items():
                pooled = pfn(h, attention_mask)
                all_emb[li][pname].append(pooled.cpu())
        
        # avg_last4: mean of layers 9, 10, 11, 12
        layers_to_avg = []
        for i in range(9, 13):
            h = captured[i]
            if h.dim() == 2:
                h = repad(h, attention_mask)
            layers_to_avg.append(h)
        h_avg = torch.stack(layers_to_avg).mean(dim=0)
        for pname, pfn in POOLING_FNS.items():
            pooled = pfn(h_avg, attention_mask)
            all_emb['avg_last4'][pname].append(pooled.cpu())
        
        all_labels.append(labels)
    
    # --- Remove hooks ---
    for h in hooks:
        h.remove()
    print(f'Removed {len(hooks)} hooks')
    
    # --- Concatenate ---
    labels_np = torch.cat(all_labels).numpy()
    embeddings_np = {}
    for layer_key in all_emb:
        embeddings_np[layer_key] = {}
        for pname in POOLING_FNS:
            t = torch.cat(all_emb[layer_key][pname], dim=0)
            embeddings_np[layer_key][pname] = t.numpy()
    
    return embeddings_np, labels_np

print('extract_all_embeddings() defined')

extract_all_embeddings() defined


## 7. Save / Load / Verify Cache

In [9]:
def save_cache(embeddings, labels, split_name, cache_dir=CACHE_DIR):
    """Save all embeddings to .npy files."""
    os.makedirs(cache_dir, exist_ok=True)
    np.save(os.path.join(cache_dir, f'{split_name}_labels.npy'), labels)
    
    count = 0
    for layer_key, pool_dict in embeddings.items():
        for pool_name, emb_array in pool_dict.items():
            fname = f'{split_name}_layer{layer_key}_{pool_name}.npy'
            np.save(os.path.join(cache_dir, fname), emb_array)
            count += 1
    
    print(f'Saved {count} embedding files + labels → {cache_dir}/')


def load_cache(split_name, layer_key, pool_name, cache_dir=CACHE_DIR):
    """Load a specific (layer, pooling) embedding + labels from cache."""
    emb = np.load(os.path.join(cache_dir, f'{split_name}_layer{layer_key}_{pool_name}.npy'))
    labels = np.load(os.path.join(cache_dir, f'{split_name}_labels.npy'))
    return emb, labels


def compute_checksum(filepath):
    """MD5 checksum for a file."""
    h = hashlib.md5()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()


def verify_cache(cache_dir=CACHE_DIR):
    """Print summary and checksums of all cached files."""
    if not os.path.exists(cache_dir):
        print('ERROR: Cache directory does not exist!')
        return
    
    files = sorted(os.listdir(cache_dir))
    total_mb = sum(os.path.getsize(os.path.join(cache_dir, f)) for f in files) / 1024**2
    print(f'Cache: {len(files)} files, {total_mb:.1f} MB total')
    print(f'Directory: {os.path.abspath(cache_dir)}')
    print()
    
    # Print checksums for reproducibility verification
    print('Checksums (MD5):')
    for f in files:
        fp = os.path.join(cache_dir, f)
        cs = compute_checksum(fp)
        size_kb = os.path.getsize(fp) / 1024
        print(f'  {cs}  {size_kb:>8.0f} KB  {f}')

print('Cache utilities defined')

Cache utilities defined


## 8. Run Extraction

This takes ~2-3 min on a 5090. After this cell, all embeddings are cached.

In [10]:
# ---- Extract TRAIN ----
print('=' * 60)
print('Extracting TRAIN embeddings...')
print('=' * 60)
train_emb, train_labels = extract_all_embeddings(
    train_loader, model, DEVICE, num_layers, desc='Train'
)
save_cache(train_emb, train_labels, 'train')

# ---- Extract TEST ----
print()
print('=' * 60)
print('Extracting TEST embeddings...')
print('=' * 60)
test_emb, test_labels = extract_all_embeddings(
    test_loader, model, DEVICE, num_layers, desc='Test'
)
save_cache(test_emb, test_labels, 'test')

Extracting TRAIN embeddings...


Train: 100%|█████████████████████████████████████████████████████████████████████████| 847/847 [01:19<00:00, 10.66it/s]


Removed 13 hooks
Saved 42 embedding files + labels → ./embedding_cache/

Extracting TEST embeddings...


Test: 100%|██████████████████████████████████████████████████████████████████████████| 283/283 [00:25<00:00, 10.98it/s]


Removed 13 hooks
Saved 42 embedding files + labels → ./embedding_cache/


## 9. Verify Cache

In [11]:
# Shape checks
print('--- Shape Verification ---')
for split in ['train', 'test']:
    for layer_key in [0, 6, 12, 'avg_last4']:
        for pool in ['mean', 'max', 'cls']:
            emb, lab = load_cache(split, layer_key, pool)
            assert emb.shape[0] == lab.shape[0], f'Mismatch! {split}/{layer_key}/{pool}'
            assert emb.shape[1] == hidden_dim, f'Wrong dim! {split}/{layer_key}/{pool}'
print(f'All shapes correct: [n_samples, {hidden_dim}]')

# Label consistency
train_lab1 = np.load(os.path.join(CACHE_DIR, 'train_labels.npy'))
train_lab2 = np.load(os.path.join(CACHE_DIR, 'train_labels.npy'))
assert np.array_equal(train_lab1, train_lab2)
print(f'Train labels: {np.bincount(train_lab1)}')

test_lab = np.load(os.path.join(CACHE_DIR, 'test_labels.npy'))
print(f'Test labels:  {np.bincount(test_lab)}')

print()
verify_cache()
# Add at the end of the verify cache cell
n_files = len([f for f in os.listdir(CACHE_DIR) if f.endswith('.npy')])
assert n_files == 86, f'Expected 86 cache files, got {n_files}!'
print(f'✓ Cache file count: {n_files}/86')

--- Shape Verification ---
All shapes correct: [n_samples, 768]
Train labels: [12355 14742]
Test labels:  [4119 4915]

Cache: 86 files, 4446.1 MB total
Directory: E:\cs566\Final_project\embedding_cache

Checksums (MD5):
  5b82b830f4260ca70ec5eeb362683db9        71 KB  test_labels.npy
  6b77259605cb77802fd0af537096f903     27102 KB  test_layer0_cls.npy
  ccf7249c9ba144c1431205a02520180d     27102 KB  test_layer0_max.npy
  358e54a88866e815b7486e78de53bc81     27102 KB  test_layer0_mean.npy
  049e50dfbfd93466bc7d1b86203d3a5e     27102 KB  test_layer10_cls.npy
  07c074b27563fae6167b25f9a78e07f2     27102 KB  test_layer10_max.npy
  dbef074613c33015c5afffd91267e1a0     27102 KB  test_layer10_mean.npy
  833aca284644341bfd65e7578712e4ad     27102 KB  test_layer11_cls.npy
  143c5efcce3ee0f86b18950a3bfa54b2     27102 KB  test_layer11_max.npy
  230a57a18ecd20b326813980e2ae5308     27102 KB  test_layer11_mean.npy
  f2f9748db2e9886a0ac3774363f6c709     27102 KB  test_layer12_cls.npy
  068dbd7e1320c

## 10. Sanity Check: Embeddings Are Non-Trivial

In [12]:
print('Mean L2 norm of train embeddings (mean pooling):')
print('-' * 50)
for layer_key in [0, 1, 3, 6, 8, 10, 12, 'avg_last4']:
    emb, _ = load_cache('train', layer_key, 'mean')
    norm = np.linalg.norm(emb, axis=1).mean()
    std  = emb.std()
    print(f'  Layer {str(layer_key):>9s}:  norm={norm:.3f},  std={std:.4f}')

print()
print('If norms/stds differ across layers → embeddings are non-trivial ✓')
print()
print('='*60)
print('✓ EXTRACTION COMPLETE — proceed to 02_linear_probe_training.ipynb')
print('='*60)

Mean L2 norm of train embeddings (mean pooling):
--------------------------------------------------
  Layer         0:  norm=0.768,  std=0.0281
  Layer         1:  norm=3.147,  std=0.1143
  Layer         3:  norm=2.715,  std=0.0986
  Layer         6:  norm=6.466,  std=0.2326
  Layer         8:  norm=10.290,  std=0.3704
  Layer        10:  norm=10.500,  std=0.3793
  Layer        12:  norm=2.515,  std=0.0908
  Layer avg_last4:  norm=6.967,  std=0.2515

If norms/stds differ across layers → embeddings are non-trivial ✓

✓ EXTRACTION COMPLETE — proceed to 02_linear_probe_training.ipynb
